Example of using GRiTS to fine-grain (i.e. backmap) a CG compound to an atomistic one.

Right now, to run this notebook you'll need to install from a fork of GRiTS.

```
git clone git@github.com:chrisjonesBSU/grits.git
cd grits
git checkout fine-grain
conda env create -f environment.yml
pip install .
conda activate grits
```

In [1]:
import warnings
warnings.filterwarnings("ignore")

import grits
from grits.finegrain import backmap_snapshot_to_compound
import gsd.hoomd
import mbuild as mb
import time

In [2]:
cg_gsd = "LJ_run_len_50_pol_50.gsd"
with gsd.hoomd.open(cg_gsd, "r") as traj:
    snap = traj[-1]

In [3]:
cg_comp = mb.load(cg_gsd)
print("Total Particles:", cg_comp.n_particles)
#cg_comp.visualize(bead_size=4).show()

Total Particles: 2500


In [4]:
bead_mapping = {"A": "c1ccc(S)cc1"} # Mapping one A bead to 1 PPS monomer
# We have to tell GRiTS which atoms on the PPS monomer are used to form monomer-monomer bonds
head_indices = {"A": [7]}
tail_indices = {"A": [10]}

start = time.time()

fg_comp = backmap_snapshot_to_compound(
    snapshot=snap,
    bead_mapping=bead_mapping,
    bond_head_index=head_indices,
    bond_tail_index=tail_indices,
    ref_distance=0.5, # The example GSD file used reduced simulation units, this puts distances back into nm.
    energy_minimize=False, # Minimize sub-groups of atomistic monomers as the fine-grain structure is being created. Still needs testing/debugging
)

end = time.time()
print("Total Time:", end - start)

2026-11-02 14:09:21,776 - mbuild.compound - WARNING - After adding new Compound, Compound.box.lengths < Compound.boundingbox.lengths. There may be particles outside of the defined simulation box
Backmapping bond 122/2450 (5%)
Backmapping bond 244/2450 (10%)
Backmapping bond 366/2450 (15%)
Backmapping bond 488/2450 (20%)
Backmapping bond 610/2450 (25%)
Backmapping bond 732/2450 (30%)
Backmapping bond 854/2450 (35%)
Backmapping bond 976/2450 (40%)
Backmapping bond 1098/2450 (45%)
Backmapping bond 1220/2450 (50%)
Backmapping bond 1342/2450 (55%)
Backmapping bond 1464/2450 (60%)
Backmapping bond 1586/2450 (65%)
Backmapping bond 1708/2450 (70%)
Backmapping bond 1830/2450 (75%)
Backmapping bond 1952/2450 (80%)
Backmapping bond 2074/2450 (85%)
Backmapping bond 2196/2450 (90%)
Backmapping bond 2318/2450 (95%)
Backmapping bond 2440/2450 (100%)
Backmapping bond 2450/2450 (100%)
Total Time: 292.16517996788025


# Next Step: Energy minimize...
# mBuild c

In [5]:
fg_comp.save("fg.mol2", overwrite=True)
fg_comp.save("fg.gsd", overwrite=True)

2026-11-02 14:14:16,262 - gmso.external.convert_hoomd - WARNING - Neither base_units or auto_scale is provided, so default units of {'length': unyt_quantity(1, 'nm'), 'energy': unyt_quantity(1, 'kJ/mol'), 'mass': unyt_quantity(1, 'amu')} are inferred.


In [ ]:
fg_comp.energy_minimize(steps=5)

In [ ]:
fg_comp.visualize().show()

In [ ]:
fg_comp.save("fg.mol2", overwrite=True)
fg_comp.save("fg.gsd", overwrite=True)